In [1]:
# Step 6 — Confirm the notebook kernel is running the project venv
import sys
import pandas as pd
import sklearn

print("executable:", sys.executable)
print("pandas    :", pd.__version__)
print("sklearn   :", sklearn.__version__)

executable: /Users/gillesgeorges/Projects/wine-quality-mlops/.venv/bin/python3
pandas    : 3.0.5
sklearn   : 1.9.0


In [2]:
# Step 7 — Load both raw CSVs, tag each with its wine type, and combine
from pathlib import Path

RAW = Path("../data/raw")

red   = pd.read_csv(RAW / "winequality-red.csv",   sep=";")
white = pd.read_csv(RAW / "winequality-white.csv", sep=";")

red["wine_type"]   = "red"
white["wine_type"] = "white"

wine = pd.concat([red, white], ignore_index=True)

print("red   :", red.shape)
print("white :", white.shape)
print("wine  :", wine.shape)
wine.head(3)

red   : (1599, 13)
white : (4898, 13)
wine  : (6497, 13)


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,wine_type
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,red
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,red
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,red


In [3]:
# Step 8 — Encode wine_type as a number, and record the mapping
WINE_TYPE_MAP = {"red": 0, "white": 1}

wine["wine_type"] = wine["wine_type"].map(WINE_TYPE_MAP)

print(wine["wine_type"].value_counts())
print()
print("dtype:", wine["wine_type"].dtype)

wine_type
1    4898
0    1599
Name: count, dtype: int64

dtype: int64


In [4]:
# Step 9 — Inspect the duplicate rows before deciding what to do with them
dup_mask = wine.duplicated()                  # True for a row seen earlier
all_copies = wine.duplicated(keep=False)      # True for every row involved

print("rows flagged as duplicates :", dup_mask.sum())
print("rows involved in any repeat:", all_copies.sum())
print("distinct wines among those :", wine[all_copies].drop_duplicates().shape[0])
print()
print("duplicates by wine type (0=red, 1=white):")
print(wine[dup_mask]["wine_type"].value_counts())
print()
print("quality mix — whole dataset vs duplicates only:")
comparison = pd.DataFrame({
    "all_rows": wine["quality"].value_counts(normalize=True).sort_index(),
    "dups_only": wine[dup_mask]["quality"].value_counts(normalize=True).sort_index(),
}).round(3)
print(comparison)

rows flagged as duplicates : 1177
rows involved in any repeat: 2169
distinct wines among those : 992

duplicates by wine type (0=red, 1=white):
wine_type
1    937
0    240
Name: count, dtype: int64

quality mix — whole dataset vs duplicates only:
         all_rows  dups_only
quality                     
3           0.005        NaN
4           0.033      0.008
5           0.329      0.328
6           0.437      0.436
7           0.166      0.189
8           0.030      0.038
9           0.001        NaN


In [6]:
# Step 10 — Drop duplicate rows, keeping the first occurrence of each wine
before = wine.shape[0]

wine = wine.drop_duplicates(keep="first").reset_index(drop=True)

after = wine.shape[0]
print("before :", before)
print("after  :", after)
print("removed:", before - after)
print()
print("remaining duplicates:", wine.duplicated().sum())
print()
print("quality counts after dedup:")
print(wine["quality"].value_counts().sort_index())

before : 6497
after  : 5320
removed: 1177

remaining duplicates: 0

quality counts after dedup:
quality
3      30
4     206
5    1752
6    2323
7     856
8     148
9       5
Name: count, dtype: int64


In [7]:
# Step 11 — Look at the extreme tails of each feature
features = [c for c in wine.columns if c not in ("quality", "wine_type")]

summary = wine[features].describe(percentiles=[0.01, 0.5, 0.99]).T
summary["max_vs_p99"] = (summary["max"] / summary["99%"]).round(1)

print(summary[["min", "1%", "50%", "99%", "max", "max_vs_p99"]].round(3))

                        min      1%      50%      99%      max  max_vs_p99
fixed acidity         3.800   5.000    7.000   12.000   15.900         1.3
volatile acidity      0.080   0.120    0.300    0.889    1.580         1.8
citric acid           0.000   0.000    0.310    0.740    1.660         2.2
residual sugar        0.600   0.900    2.700   18.150   65.800         3.6
chlorides             0.009   0.021    0.047    0.200    0.611         3.1
free sulfur dioxide   1.000   3.190   28.000   76.000  289.000         3.8
total sulfur dioxide  6.000  10.000  116.000  240.000  440.000         1.8
density               0.987   0.989    0.995    1.001    1.039         1.0
pH                    2.720   2.890    3.210    3.650    4.010         1.1
sulphates             0.220   0.300    0.510    1.020    2.000         2.0
alcohol               8.000   8.700   10.400   13.400   14.900         1.1


In [8]:
# Step 12 — Inspect the rows behind the two most extreme values
cols = ["residual sugar", "free sulfur dioxide", "total sulfur dioxide",
        "density", "alcohol", "quality", "wine_type"]

print("Top 3 by residual sugar:")
print(wine.nlargest(3, "residual sugar")[cols])
print()
print("Top 3 by free sulfur dioxide:")
print(wine.nlargest(3, "free sulfur dioxide")[cols])

Top 3 by residual sugar:
      residual sugar  free sulfur dioxide  total sulfur dioxide  density  \
3653           65.80                  8.0                 160.0  1.03898   
2753           31.60                 35.0                 176.0  1.01030   
4314           26.05                 27.0                 122.0  1.00295   

      alcohol  quality  wine_type  
3653     11.7        6          1  
2753      8.8        6          1  
4314     10.6        6          1  

Top 3 by free sulfur dioxide:
      residual sugar  free sulfur dioxide  total sulfur dioxide  density  \
5186             2.9                289.0                 440.0  0.99314   
2981             2.0                146.5                 307.5  0.99240   
3866             1.7                138.5                 272.0  0.99452   

      alcohol  quality  wine_type  
5186     10.5        3          1  
2981     11.0        3          1  
3866      9.6        4          1  


## Data preparation decisions

**Duplicates — dropped (6,497 → 5,320 rows).**
1,177 exact duplicate rows removed, keeping the first occurrence.
992 distinct wines were involved, averaging 2.19 copies each — collisions
at measurement precision, not a broken export. Duplication rate was 18–23%
across quality 5–8 and **0% at quality 3 and 9**, so the scarce extremes
were untouched. Dropping removes leakage risk (the same wine appearing in
both train and holdout) without reshaping the target distribution.

**Outliers — kept, all of them.**
Four features have a max more than 3x their 99th percentile. Both were
checked at row level and are internally consistent:
- Residual sugar 65.8 g/L is also the densest wine (1.039) — sugar raises
  density, so two columns agree. Ordinary dessert wine.
- Free SO2 289 mg/L belongs to a **quality-3** wine, as do the next two
  most-sulfured wines (146.5 → quality 3, 138.5 → quality 4). Excess SO2
  is *why* they score badly; the extreme value is signal, not error.

Capping would flatten the strongest explanatory signal in our rarest class
(only 30 quality-3 wines exist). Random Forest is threshold-based and
unaffected by extreme values; any capping would also have to be replicated
exactly in the Lambda at inference time — another silent-failure contract.

In [9]:
# Step 14 — Build the stratification bins and confirm they're healthy
RANDOM_STATE = 42

strata = wine["quality"].clip(lower=4, upper=8)

print("original quality counts:")
print(wine["quality"].value_counts().sort_index())
print()
print("stratification bins (3->4, 9->8):")
print(strata.value_counts().sort_index())
print()
print("quality column unchanged:", sorted(wine["quality"].unique()))

original quality counts:
quality
3      30
4     206
5    1752
6    2323
7     856
8     148
9       5
Name: count, dtype: int64

stratification bins (3->4, 9->8):
quality
4     236
5    1752
6    2323
7     856
8     153
Name: count, dtype: int64

quality column unchanged: [np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9)]


In [11]:
# Step 15 — Split into train (60%) / validation (20%) / holdout (20%)
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

wine["strat_bin"] = wine["quality"].clip(lower=4, upper=8)

train_val, holdout    = train_test_split(wine,      test_size=0.20,
                                         stratify=wine["strat_bin"],      random_state=RANDOM_STATE)
train,     validation = train_test_split(train_val, test_size=0.25,
                                         stratify=train_val["strat_bin"], random_state=RANDOM_STATE)

for name, df in [("train", train), ("validation", validation), ("holdout", holdout)]:
    print(f"{name:11s} {df.shape[0]:5d} rows")

print()
print("total:", len(train) + len(validation) + len(holdout), "of", len(wine))

train        3192 rows
validation   1064 rows
holdout      1064 rows

total: 5320 of 5320


In [12]:
# Step 16 — Save the three splits to data/processed/
PROC = Path("../data/processed")
PROC.mkdir(parents=True, exist_ok=True)

for name, df in [("train", train), ("validation", validation), ("holdout", holdout)]:
    out = df.drop(columns="strat_bin")
    path = PROC / (name + ".csv")
    out.to_csv(path, index=False)
    print(name, "->", path, out.shape)

train -> ../data/processed/train.csv (3192, 13)
validation -> ../data/processed/validation.csv (1064, 13)
holdout -> ../data/processed/holdout.csv (1064, 13)


In [13]:
# Show where the notebook is running from
import os
print(os.getcwd())
print(PROC.resolve())

/Users/gillesgeorges/Projects/wine-quality-mlops/notebooks
/Users/gillesgeorges/Projects/wine-quality-mlops/data/processed
